# Day 2 · 02. 재순위화 (Reranking)

## 학습 목표
- Bi-encoder와 Cross-encoder의 구조·속도·품질 차이를 설명할 수 있다.
- BGE-reranker-v2-m3(🔒)를 사용해 기존 retriever의 상위 K 결과를 재정렬할 수 있다.
- 이전 노트북의 **하이브리드(BM25 + Dense)** 를 1단계로 쓰고, 그 위에 reranker를 얹어 **완성형 2-stage retrieval** 파이프라인을 구현할 수 있다.
- 전자금융 표준약관 벤치마크 5개 질문으로 **hybrid vs hybrid+rerank** 결과를 비교·평가할 수 있다.

## 핵심 키워드
Bi-encoder · Cross-encoder · BGE-reranker-v2-m3 · 2-stage retrieval · Hybrid + Rerank · NDCG

## 이 노트북의 위치

```
1단계 (이전 노트북 01)              2단계 (이 노트북)
Hybrid (Dense + BM25, RRF)   ─▶   Cross-Encoder Reranker
└─ "넓게 후보를 모은다"              └─ "정답을 꼭대기에 올린다"
       recall 확보                      precision 확보
```

`01_hybrid_ensemble.ipynb` 에서 만든 **하이브리드 retriever가 이 노트북의 1단계 입력**이다. 여기서는 그 상위 K 후보를 Cross-Encoder로 다시 채점해 최종 top-N을 골라낸다. Anthropic Contextual Retrieval 실험(2024)에서 dense 단독 대비 **실패율 67% 감소**로 보고된 바로 그 구성이다.

## 사전 준비
- `data/corpus_ko.txt` 로부터 하이브리드 retriever(FAISS + BM25)와 reranker를 **본 노트북 내에서 즉석 구축**한다. 01 노트북을 먼저 실행하지 않아도 독립적으로 돌아간다.

In [1]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '..')

from common import get_chat_model, get_embeddings, provider_badge
from common.ko_tokenizer import tokenize_ko
from common.loaders import load_any

print(provider_badge())


☁️ OpenAI | model=openai/gpt-4o-mini


## 1. 왜 재순위화가 필요한가?

1단계 검색기(하이브리드든 dense 단독이든)는 공통적으로 **질문과 문서를 각자 따로 인코딩**한다. BM25는 토큰 빈도로, dense는 벡터 유사도로 점수를 매기지만 둘 다 *질문과 문서 토큰 간의 상호작용* 을 보지 않는다. 아래 두 인코딩 방식의 차이를 먼저 짚어보자.

### Bi-encoder (🔒 빠름, 품질 중간)
```
질문 ─▶ [Encoder] ─▶ q_vec ┐
                              ├─ cosine ─▶ score
문서 ─▶ [Encoder] ─▶ d_vec ┘
```
- 문서 임베딩을 **오프라인에서 미리** 계산해 저장 → 대규모 검색에 유리.
- 질의 시 1회의 인코딩 + 수백만 건의 벡터 내적(ANN 인덱스로 O(log N) 수준)만 수행.
- 단점: 질문과 문서를 독립적으로 인코딩하므로 **상호작용 정보**가 없다. "망분리"라는 단어가 겹치기만 해도 높은 점수를 얻을 수 있어 **표면적 유사도**에 취약하다.

### Cross-encoder (🔒 느림, 품질 높음)
```
[CLS] 질문 [SEP] 문서 [SEP] ─▶ [Encoder] ─▶ 관련성 score (스칼라)
```
- 질문과 문서를 **한 시퀀스로 연결해 함께** 인코딩 → 모든 토큰 쌍 간 self-attention으로 세밀한 의미 매칭이 가능.
- "질문이 묻는 **무엇**"과 "문서가 말하는 **무엇**"을 토큰 레벨에서 비교하므로, 단순 키워드 중복이 아니라 *실제 답을 담고 있는지*를 판별할 수 있다.
- 단점: 모든 후보 (질문, 문서) 쌍에 대해 모델 forward를 1번씩 돌려야 해서 비용이 O(K). 100만 건 전체에 적용하는 것은 비현실적.

### 실무 해결: 2-stage retrieval

| 단계 | 방식 | 후보 수 | 속도 | 역할 |
|------|------|--------|------|------|
| 1단계 | **Hybrid (Dense + BM25 + RRF)** | top-20~100 | 빠름 | **recall** 확보 (정답이 후보에 "들어오게") |
| 2단계 | Cross-encoder (rerank) | top-5~10 | 느림 | **precision** 확보 (정답을 "위로 올리기") |

> 💡 **직관**: 1단계는 "관련 있어 보이는 20~100개를 빠르게 긁어오고", 2단계는 "그중 정말 관련 있는 5~10개를 꼼꼼히 고른다". 넓은 그물로 물고기를 퍼올린 뒤(recall) 상품 가치가 있는 것만 고르는(precision) 어시장 방식과 같다.

> 📌 **왜 1단계를 하이브리드로 쓰는가?** Reranker는 *이미 검색된 후보만* 재정렬할 수 있다. Dense 단독은 "전자금융감독규정" 같은 **고유명사·희귀 전문용어**를 놓칠 수 있는데, 이렇게 놓친 정답은 reranker가 아무리 강력해도 복구할 수 없다. BM25와의 하이브리드는 이 누락을 막아 reranker에게 **재정렬할 가치가 있는 후보 풀**을 안정적으로 공급한다.

### 언제 reranker를 쓰지 말아야 하는가?
- **LLM 컨텍스트가 충분하고 top-5 정확도가 이미 높다면** → 추가 지연을 감수할 이유가 적다.
- **실시간성이 극단적으로 중요한 경우** (예: 자동완성) → 100ms의 rerank 비용이 부담.
- **후보 문서가 매우 길다면** → cross-encoder 입력 길이 제한(보통 512 토큰)으로 잘려서 이점이 사라질 수 있다. 청크 단위가 적절한지 먼저 점검.


## 2. 1단계 하이브리드 retriever 구축

`data/corpus_ko.txt` 를 로드해 1단계 recall 확보용 **하이브리드 retriever**(FAISS dense + BM25 sparse, RRF 결합)를 구성한다. 이 구성은 이전 노트북 `01_hybrid_ensemble.ipynb` 에서 다룬 패턴을 그대로 가져온 것이다.

- `chunk_size=300, chunk_overlap=50`: 한국어 법령·규정 문장 기준으로 한두 문장 단위의 의미 단위를 유지하는 값. 너무 크면 rerank 입력이 늘어 속도 손해, 너무 작으면 맥락이 잘린다.
- `k=20`: 2단계 rerank가 충분한 후보 풀을 보도록 1단계에서 **과잉 검색(over-retrieve)** 한다. 최종 출력이 top-5라면 1단계는 최소 그 **4배 이상**을 뽑는 것이 관례(Anthropic 실험에서는 top-150 → top-20).
- `tokenize_ko` (kiwi 형태소 분석기): 한국어 BM25에서 "전자금융거래는/가/의"를 같은 어간으로 묶기 위한 필수 전처리.


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_core.documents import Document

# (1) 기본 코퍼스 로드 및 청킹
docs = load_any('../data/corpus_ko.txt')
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
base_chunks = splitter.split_documents(docs)

# (2) distractor 청크 — 질문당 3종씩 총 15개.
#     gold와 비슷한 길이(≈200자)에 질문 키워드를 높은 밀도로 담아 BM25·dense 모두 혼동시키고,
#     cross-encoder만 정답을 식별할 수 있는 조건을 만든다.
DISTRACTORS = [
    # ===== Q1 함정: 대면·자동화·금융상품·거래·법 키워드 공유 =====
    '금융회사의 일반적 대면 창구 업무는 종이 서류 기반으로 이루어지며, 고객의 신원을 즉석에서 확인하는 절차를 따른다. 이는 전자금융과 구분되는 전통적 금융 거래 방식이다.',
    '방문판매 등에 관한 법률은 판매업자가 소비자와 직접 대면하지 않고 전화·ARS 등 자동화된 응답 체계로 금융상품을 판매하는 거래 방식에 별도 고지 의무를 정한다. 이러한 비대면 판매 방식은 일반 매장 판매와 구분해 특수판매라고 부른다.',
    '여신전문금융업법상 신용카드업자는 가맹점에서의 대면 거래와 인터넷·모바일 앱 기반의 자동화된 비대면 거래를 구분 관리하며, 거래 유형마다 수수료 산정 방식이 달리 적용되는 금융상품 이용 규정을 법으로 정한다.',
    # ===== Q2 함정: 살아 있는 개인·식별·정보·법·정의 키워드 공유 =====
    '통계 작성을 목적으로 수집된 집계 정보는 개별 개인을 식별할 수 없도록 가공한 뒤에만 공개할 수 있다. 통계법은 이러한 공공 통계의 산출과 공표 절차를 정한다.',
    '신용정보의 이용 및 보호에 관한 법률은 살아 있는 개인이나 법인의 신용도를 판단할 수 있는 정보를 신용정보로 정의하며, 이는 개인정보와 구분되는 별도의 법적 정의 체계를 이룬다. 성명·주민등록번호를 포함한 식별 요소도 함께 다룬다.',
    '정보통신망법 제2조는 온라인 서비스에서 통신·게시물·계정을 통해 이용자를 식별할 수 있는 정보의 처리 기준을 정하며, 살아 있는 자연인에 관한 정보 중 일부 영역에 한정해 규율한다는 점에서 다른 법률의 정의와 경계를 달리한다.',
    # ===== Q3 함정: 신용정보·고객·수집·처리·위탁·관리·의무·법률 키워드 공유 =====
    '금융회사는 자금세탁 방지를 위해 고객확인 의무와 의심거래 보고 의무를 이행한다. 이는 특정금융거래정보의 보고 및 이용 등에 관한 법률에 근거한 별개의 규제 체계다.',
    '자금세탁방지법상 금융회사는 고객확인 절차에서 신용정보를 포함한 고객 정보를 수집·처리하며, 외부 위탁자에게 업무를 맡길 때에도 고객확인 의무가 면제되지 않는 관리 책임을 진다. 이 법률이 정한 수집 범위는 한정적이다.',
    '전자금융거래법은 전자금융업자가 이용자 신용정보 등 고객 정보를 수집·처리하는 과정에서 수탁자에 대한 관리·감독 의무를 별도로 규정한다. 전자금융감독규정과 더불어 수집 범위의 허용 한계를 정한 조문이 포함된 대표적 법률이다.',
    # ===== Q4 함정: 전자금융감독규정·전산실·정보처리시스템·운영·분리 키워드 공유 =====
    '클라우드 환경에서는 가상 사설 클라우드(VPC)와 보안 그룹을 활용해 트래픽을 논리적으로 분할한다. 이는 물리적 네트워크 분리와는 구분되는 소프트웨어 기반 격리 기법이다.',
    '클라우드 환경으로 금융회사 전산실과 정보처리시스템을 이전할 때에는 전자금융감독규정 가이드라인에 따라 VPC와 보안 그룹을 통한 논리 격리 운영 환경이 권고되며, 이는 물리적 망 분리와 구별되는 별도의 통제 접근이다.',
    '정보보호관리체계(ISMS) 인증은 전자금융감독규정과 별개 체계로 전산실·정보처리시스템의 물리적 접근통제와 운영 문서화를 요구한다. 감독규정이 요구하는 망 구분과는 다른 통제 영역의 운영 환경 기준이다.',
    # ===== Q5 함정: 그래프·관계·다단계 추론·요약 질의·벡터 검색 키워드 공유 =====
    '관계형 데이터베이스는 행과 열로 구성된 테이블에 데이터를 저장하며 외래 키로 테이블 간 연결을 표현한다. 그래프 구조와 달리 다단계 추론보다는 정형 질의에 최적화되어 있다.',
    '전통적인 검색엔진은 역색인(inverted index) 기반 접근으로 키워드 매칭을 빠르게 수행하지만, 벡터 검색이나 그래프 기반 다단계 추론이 요구되는 요약 질의에서는 의미 파악이 부족해 RAG 파이프라인의 1단계로만 활용되는 경우가 많다.',
    '속성 그래프(Property Graph)는 노드와 엣지에 속성 값을 부여해 데이터 관계를 표현하지만, 쿼리 언어(Cypher 등) 최적화 정도에 따라 다단계 추론과 요약 질의 성능이 제각각이다. 벡터 검색 기반 파이프라인보다 항상 우수하다고 단정할 수는 없다.',
]
distractor_chunks = [Document(page_content=t, metadata={'source': 'distractor'}) for t in DISTRACTORS]

# (3) 전체 청크 = 실제 코퍼스 + distractor
chunks = base_chunks + distractor_chunks
print(f'실제 코퍼스 청크: {len(base_chunks)} + distractor: {len(distractor_chunks)} = 전체 {len(chunks)}')

# (A) Dense retriever (FAISS)
vectordb = FAISS.from_documents(chunks, get_embeddings())
dense = vectordb.as_retriever(search_kwargs={'k': 20})

# (B) Sparse retriever (BM25 + kiwi 형태소)
sparse = BM25Retriever.from_documents(chunks, preprocess_func=tokenize_ko)
sparse.k = 20

# (C) 하이브리드: Dense + BM25를 RRF로 결합 → 1단계 base_retriever
base_retriever = EnsembleRetriever(
    retrievers=[dense, sparse],
    weights=[0.5, 0.5],
)
print('1단계 하이브리드 retriever 구축 완료 (FAISS + BM25, top-20)')


실제 코퍼스 청크: 4 + distractor: 15 = 전체 19
1단계 하이브리드 retriever 구축 완료 (FAISS + BM25, top-20)


## 3. BGE-reranker-v2-m3 로드 🔒

[BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3)는 한국어·영어·중국어를 포함한 100+개 언어를 지원하는 다국어 cross-encoder 모델이다. 이름의 `m3`는 **Multi-Linguality, Multi-Functionality, Multi-Granularity** 를 의미하며, 문장/문단/문서 어느 단위에서도 안정적으로 동작하도록 학습됐다.

### 핵심 특징
- **XLM-RoBERTa large 기반** (~568M 파라미터): GPU 없이도 CPU/FP16으로 실행 가능.
- **입력 길이 8192 토큰**: 일반 cross-encoder(512)보다 길어 법령·약관 같은 긴 청크에 유리.
- `normalize=True` 옵션: raw logit을 시그모이드로 0~1 범위 확률 유사 점수로 변환 → 임계값 설정이 쉬움.

### 점수 해석 가이드 (경험칙)
| 점수 범위 | 의미 |
|----------|------|
| > 0.8 | 질문에 직접적 답을 포함 (높은 신뢰) |
| 0.4 ~ 0.8 | 관련은 있으나 부분적·간접적 |
| < 0.4 | 잡음에 가까움 — 상위 K에서 제외 고려 |

> ⚠️ 최초 실행 시 약 2.3GB 모델을 다운로드한다. 폐쇄망에서는 `HF_HOME`에 사전 반입이 필요하다. 사내 모델 허브가 있다면 `local_files_only=True` 로 로컬 경로를 지정해 쓸 수 있다.


In [3]:
from FlagEmbedding import FlagReranker

# normalize=True로 설정하면 0~1 사이 확률 유사 점수로 변환
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

# 간단 테스트: 질문-문서 페어에 대한 점수
pairs = [
    ('망분리란 무엇인가?', '망분리는 업무망과 인터넷망을 분리해 보안을 강화한다.'),
    ('망분리란 무엇인가?', 'LLM은 Transformer 아키텍처 기반의 모델이다.'),
]
scores = reranker.compute_score(pairs, normalize=True)
for (q, d), s in zip(pairs, scores):
    print(f'score={s:.4f} | {d[:40]}...')


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


score=0.9647 | 망분리는 업무망과 인터넷망을 분리해 보안을 강화한다....
score=0.0000 | LLM은 Transformer 아키텍처 기반의 모델이다....


## 4. LangChain ContextualCompressionRetriever에 reranker 연결

LangChain은 "1단계 검색 → 후처리(압축/필터링/재정렬)" 패턴을 **`ContextualCompressionRetriever`** 로 추상화해 제공한다. 이름은 "compression"이지만 실제로는 *후보 집합을 줄이거나 재배열하는 모든 연산* 을 포괄한다 — reranking도 그중 하나다.

```
사용자 query
    │
    ▼
base_retriever  ──▶ top-20 후보
  ├─ FAISS (dense)
  └─ BM25  (sparse)  ← RRF로 결합
    │
    ▼
base_compressor (BGE reranker) ──▶ top-5 재정렬된 결과
```

`BaseDocumentCompressor`를 상속해 `compress_documents(documents, query)` 메서드만 구현하면 된다. 이 인터페이스 덕분에 reranker를 다른 LangChain 체인(RAG, agent 등)에 **투명하게 끼워 넣을 수 있다**. 재순위 점수는 `doc.metadata['rerank_score']` 에 기록해 두면 디버깅·모니터링에 유용하다.

In [4]:
from typing import Sequence
from langchain_core.documents import Document
from langchain_core.callbacks import Callbacks
from langchain.retrievers import ContextualCompressionRetriever
from langchain_core.documents.compressor import BaseDocumentCompressor

class BgeRerankCompressor(BaseDocumentCompressor):
    '''BGE reranker를 LangChain compressor 인터페이스로 감싼 클래스.'''
    model_name: str = 'BAAI/bge-reranker-v2-m3'
    top_n: int = 5

    class Config:
        arbitrary_types_allowed = True

    def compress_documents(
        self,
        documents: Sequence[Document],
        query: str,
        callbacks: Callbacks = None,
    ) -> Sequence[Document]:
        if not documents:
            return []
        pairs = [(query, d.page_content) for d in documents]
        scores = reranker.compute_score(pairs, normalize=True)
        ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
        top = ranked[: self.top_n]
        # 점수를 메타데이터에 기록
        out = []
        for d, s in top:
            d.metadata['rerank_score'] = float(s)
            out.append(d)
        return out

compressor = BgeRerankCompressor(top_n=5)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)
print('2-stage retriever 준비 완료 (hybrid top-20 → rerank top-5)')


2-stage retriever 준비 완료 (hybrid top-20 → rerank top-5)


/tmp/ipykernel_49818/653421992.py:7: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class BgeRerankCompressor(BaseDocumentCompressor):


### 사용 예시: hybrid top-20 → rerank top-5 단계별로 보기

벤치마크로 넘어가기 전에, 방금 만든 `base_retriever`(하이브리드)와 `rerank_retriever`(하이브리드 + BGE rerank)가 **같은 질문을 어떻게 다르게 정렬**하는지 한 번 눈으로 확인해보자.

아래 셀은 질문 하나(`"대면 없이 자동화된 방식으로 금융상품을 이용하는 거래를 법에서 뭐라고 부르는가?"`)를 가지고,

1. **1단계 하이브리드**로 top-20 후보를 뽑아 순위별로 출력하고,
2. 그 top-20을 **2단계 BGE reranker**에 통과시킨 top-5를 `rerank_score`와 함께 보여준 뒤,
3. reranker로 고른 문서가 *원래 하이브리드에서 몇 위였는지*(rank shift)를 짝지어 보여준다.

함정으로 심어둔 **distractor가 상위권을 점령했다가**, reranker가 실제 정답(`'전자금융거래는 금융회사…'` 가 들어간 청크)을 몇 계단 끌어올리는지 관찰해 보자.

In [ ]:
sample_q = '대면 없이 자동화된 방식으로 금융상품을 이용하는 거래를 법에서 뭐라고 부르는가?'

# (1) 1단계 하이브리드 top-20
hybrid_docs = base_retriever.invoke(sample_q)
print(f'[1단계 hybrid top-{len(hybrid_docs)}]  질문: {sample_q}\n' + '-' * 80)
for i, d in enumerate(hybrid_docs, start=1):
    src = d.metadata.get('source', '-')
    tag = 'distractor' if src == 'distractor' else 'corpus'
    preview = d.page_content.replace('\n', ' ')[:60]
    print(f'  {i:>2}. [{tag:>10}] {preview}...')

# (2) 2단계 rerank top-5 (내부적으로 위 hybrid top-20을 재정렬)
rerank_docs = rerank_retriever.invoke(sample_q)
print(f'\n[2단계 rerank top-{len(rerank_docs)}]\n' + '-' * 80)

# (3) rerank 결과 문서가 hybrid에서 몇 위였는지 역추적
hybrid_texts = [d.page_content for d in hybrid_docs]
for i, d in enumerate(rerank_docs, start=1):
    score = d.metadata.get('rerank_score', float('nan'))
    try:
        prev_rank = hybrid_texts.index(d.page_content) + 1
        shift = f'hybrid {prev_rank:>2}위 → rerank {i}위'
    except ValueError:
        shift = f'hybrid 밖 → rerank {i}위'
    src = d.metadata.get('source', '-')
    tag = 'distractor' if src == 'distractor' else 'corpus'
    preview = d.page_content.replace('\n', ' ')[:60]
    print(f'  {i}. score={score:.4f}  [{tag:>10}]  ({shift})')
    print(f'     {preview}...')

## 5. 벤치마크 5개 질문으로 hybrid vs hybrid+rerank 비교

이제 reranker가 **하이브리드 검색 결과 위에서도 추가 이득**을 주는지 정량적으로 검증한다. 단순히 dense 단독과 비교하는 것은 공정하지 않다 — 현실에서 1단계는 이미 하이브리드일 가능성이 높으니, **hybrid vs hybrid+rerank** 로 비교해야 "reranker의 마지막 1마일 효과"를 제대로 측정할 수 있다.

### 질문 난이도가 메트릭에 미치는 영향
벤치마크 설계에는 함정이 하나 있다. 질문이 **지문 문장을 거의 그대로 옮긴 수준**이면 BM25만으로도 정답 청크가 1위로 올라와 reranker가 개입할 여지가 없다. 그러면 "개선 = 0" 이 양산되고, 그래프만 보면 reranker가 쓸모없어 보인다 — 실제로는 질문이 너무 쉬워서다.

이 노트북의 벤치마크는 reranker가 **실제로 쓸모 있는 구간**을 겨냥해 다음 세 가지 함정을 의도적으로 심어놓았다.

1. **표현 거리(Paraphrase stress)** — 질문에 정답 청크의 핵심 명사(예: "전자금융거래")를 직접 쓰지 않는다. BM25의 exact-match 이점을 상쇄해 하이브리드 1단계 순위를 흔든다.
2. **키워드 혼선(Distractor overlap)** — 질문 표현이 *정답이 아닌 다른 청크*와도 표면 키워드(예: "법", "정보")를 공유한다. Dense 유사도도 헷갈린다.
3. **동일 주제 분기(Same-topic siblings)** — "지식 그래프 vs 벡터 검색 vs RAG vs LLM" 처럼 인접 주제 청크가 여러 개 있어, 단어 매칭만으로는 오답으로 빠지기 쉬운 상황을 만든다.

이 조건에서 **cross-encoder의 토큰 레벨 상호작용**이 비로소 위력을 발휘한다 — 표면 단어는 겹쳐도 "질문이 묻는 것"과 "문서가 답하는 것" 의 매칭을 세밀하게 읽어내기 때문이다.

### 평가 방식: gold chunk의 rank + MRR@5
- 질문마다 "정답이 들어있는 청크"를 식별할 수 있는 **gold snippet**(짧은 고유 구절)을 지정한다.
- 검색 결과에서 gold snippet을 포함한 문서의 **순위(rank)** 를 측정한다 — 1위면 완벽, 6위 이상이면 top-5 밖 "miss".
- **MRR(Mean Reciprocal Rank)** = `1 / rank` 의 평균. 정답을 위로 올릴수록 급격히 점수가 올라 **"얼마나 위로 끌어올렸는가"** 를 순위 민감하게 반영한다.

> 📌 **왜 hit-rate 대신 rank/MRR 인가?**  
> 정답 청크가 1위든 5위든 hit-rate@5는 둘 다 `1.0` 으로 같다. 하지만 실제 LLM은 상위 문서에 훨씬 더 많이 주목한다 — 같은 1.0 이라도 1위인 편이 RAG 답변 품질에 결정적이다. Rank/MRR은 이 **위치 민감도**를 그대로 보여준다.

### 비교 지점

| 관찰 지점 | 무엇을 말해주는가 |
|---|---|
| **gold의 hybrid top-20 순위** | 1단계가 정답을 후보 풀에 넣긴 넣었는지 (recall 지표) |
| **hybrid@5 순위** vs **rerank@5 순위** | 2단계가 정답을 얼마나 위로 끌어올렸는지 (precision 지표) |
| **MRR 개선폭** | 5개 질문 평균으로 본 reranker의 체감 효용 |

**해석 가이드**
- **큰 양수 ΔRR**: 하이브리드는 정답을 후보에 넣어줬지만 3~5위에 묻어뒀고, reranker가 이를 1~2위로 끌어올린 경우. 여기서 reranker의 진짜 가치가 드러난다.
- **0 근처**: 하이브리드가 이미 정답을 1위로 올린 경우. Reranker는 확인만 해준 셈. (이 상태가 *모든* 질문에서 나오면 벤치마크가 너무 쉬운 것이니 질문을 다시 설계해야 한다.)
- **음수(악화)**: Reranker가 표면 유사해 보이는 잡음 문서를 잘못 올린 경우. 후보 풀 크기(K=20)나 청크 전략 재검토가 필요.


In [5]:
# 난이도 있는 벤치마크 — 세 가지 함정(표현 거리·키워드 혼선·동일 주제 분기)을 의도적으로 심었다.
# 각 질문마다 정답 청크를 식별할 수 있는 'gold_snippet'(짧은 고유 구절)을 지정한다.
BENCHMARK = [
    {
        # ① 표현 거리: '전자금융거래' 라는 정답 키워드를 질문에 쓰지 않는다 → BM25 exact-match 무력화
        'q': '대면 없이 자동화된 방식으로 금융상품을 이용하는 거래를 법에서 뭐라고 부르는가?',
        'gold_snippet': '전자금융거래는 금융회사',
    },
    {
        # ② 키워드 혼선: '법' · '정보' 만으로는 신용정보법·전자금융법 청크와 섞여 순위가 흔들린다
        'q': '살아있는 개인을 식별할 수 있는 정보의 법적 정의는 무엇인가?',
        'gold_snippet': '개인정보보호법 제2조',
    },
    {
        # ③ 동일 주제 분기: '정보' · '법' · '금융회사' 3중 키워드가 개인정보/전자금융 청크와 겹침
        'q': '고객 신용정보 처리 시 수집 범위와 위탁자 관리 의무를 정한 법률은?',
        'gold_snippet': '신용정보의 이용 및 보호',
    },
    {
        # ④ 질문에 '분리'가 없다 — dense 임베딩이 표면 유사도로만 판단하면 놓치기 쉬움
        'q': '전자금융감독규정이 요구하는 전산실·정보처리시스템의 운영 환경은?',
        'gold_snippet': '업무망과 인터넷망',
    },
    {
        # ⑤ LLM · RAG · GraphRAG 세 청크가 공통 단어를 공유 — cross-encoder 변별력 시험
        'q': '벡터 검색보다 다단계 추론과 요약 질의에 유리한 그래프 기반 접근은?',
        'gold_snippet': '지식 그래프는 개체(Entity)와 관계(Relation)',
    },
]

MISS_RANK = 99   # gold가 결과에 없을 때 부여하는 "매우 낮은" 순위 (RR=0에 해당)

def rank_of(docs, gold_snippet: str) -> int:
    '''docs 리스트에서 gold_snippet을 포함한 첫 문서의 순위(1-indexed). 없으면 MISS_RANK.'''
    for i, d in enumerate(docs, start=1):
        if gold_snippet in d.page_content:
            return i
    return MISS_RANK

def rr(rank: int) -> float:
    '''Reciprocal Rank: 1위=1.0, 2위=0.5, 3위≈0.33, miss=0.0'''
    return 1.0 / rank if rank < MISS_RANK else 0.0

def fmt_rank(r: int) -> str:
    return str(r) if r < MISS_RANK else '밖'


In [6]:
import pandas as pd

rows = []
h_rr, r_rr = [], []   # 질문별 reciprocal rank 누적 (MRR 계산용)

for item in BENCHMARK:
    q, gold = item['q'], item['gold_snippet']

    # 1단계 하이브리드 전체 후보(top-20) — reranker가 다룰 재료 풀
    hybrid20 = base_retriever.invoke(q)
    r_h20 = rank_of(hybrid20, gold)

    # 1단계만 그대로 top-5로 자른 경우의 gold 순위
    r_h5  = rank_of(hybrid20[:5], gold)

    # 2-stage: hybrid top-20 → BGE reranker → top-5 재정렬
    rerank5 = rerank_retriever.invoke(q)
    r_r5  = rank_of(rerank5, gold)

    h_rr.append(rr(r_h5))
    r_rr.append(rr(r_r5))

    rows.append({
        '질문': q[:30] + ('…' if len(q) > 30 else ''),
        'gold의 hybrid@20 순위': fmt_rank(r_h20),
        'hybrid@5 순위': fmt_rank(r_h5),
        'rerank@5 순위': fmt_rank(r_r5),
        'ΔRR (rerank − hybrid)': f'{rr(r_r5) - rr(r_h5):+.3f}',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

mrr_h = sum(h_rr) / len(h_rr)
mrr_r = sum(r_rr) / len(r_rr)
print(f'\n평균 MRR@5  —  hybrid          : {mrr_h:.3f}')
print(f'평균 MRR@5  —  hybrid + rerank : {mrr_r:.3f}')
print(f'개선 폭                        : {mrr_r - mrr_h:+.3f}')


                             질문 gold의 hybrid@20 순위 hybrid@5 순위 rerank@5 순위 ΔRR (rerank − hybrid)
대면 없이 자동화된 방식으로 금융상품을 이용하는 거래를…                  4           4           2                +0.250
살아있는 개인을 식별할 수 있는 정보의 법적 정의는 무…                  3           3           2                +0.167
고객 신용정보 처리 시 수집 범위와 위탁자 관리 의무를…                  3           3           1                +0.667
전자금융감독규정이 요구하는 전산실·정보처리시스템의 운영…                  4           4           3                +0.083
벡터 검색보다 다단계 추론과 요약 질의에 유리한 그래프…                  3           3           2                +0.167

평균 MRR@5  —  hybrid          : 0.300
평균 MRR@5  —  hybrid + rerank : 0.567
개선 폭                        : +0.267
